# ADIM 8 — Kâr-odaklı baseline: ProfLogit vs bizim yaklaşım

"Kâr odaklıysan neden kâr-odaklı eğitilen modelle kıyaslamadın?" itirazını kapatır.
İki yaklaşım (5 set, stratified 5-kat, seed=42, aynı CLV/kâr parametreleri, aynı encode+scale):
**(A) Bizimki** = ham LightGBM (Adım 2 best_params) + kâr-maksimize eşik t*;
**(B) ProfLogit** = lojistik katsayılarını doğrudan **EMPC-maksimize** eden model
(gerçek-kodlu genetik arama, warm-start = MLE lojistik). Tüm fit/eşik yalnız fold-içi.
ProfLogit bilimsel atıf: Stripling ve ark. (2018). Ağır mantık `src/proflogit_baseline.py`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml bulunamadı")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from scipy.stats import wilcoxon

from src import config as cfg
from src import plotstyle as ps
from src import proflogit_baseline as pb
from src import strings_tr as S

ps.uygula()
cfg.klasorleri_hazirla()
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 40)

CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))


yaz(S.MSG8["atif"])

ProfLogit yöntemi: Stripling, vanden Broucke, Antonio, Baesens, Snoeck (2018), 'Profit maximizing logistic model for customer churn prediction using genetic algorithms' (EMPC amaç fonksiyonu + genetik katsayı arama).


## 1. Her set: iki yaklaşımı 5-kat çalıştır

In [2]:
import time

veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}
yaz(S.MSG["bolum"].format(ad="PROFLOGIT vs BİZİMKİ"))
tum = {}
for k in cfg.DATASETS:
    t = time.time()
    met, n, N = pb.calistir_set(k, veriler[k], cfg.SEED)
    tum[k] = (met, n, N)
    if N > n:
        yaz(S.MSG8["ornek"].format(set=k, n=n, N=N, seed=cfg.SEED))
    for adi in ("ours", "proflogit"):
        m = met[adi]
        yaz(S.MSG8["set"].format(set=k, yaklasim=S.YAKLASIM_AD[adi], empc=np.mean(m["EMPC"]),
            kar=np.mean(m["kar"]), roi=np.mean(m["roi"]), pr=np.mean(m["PR-AUC"]),
            sure=(time.time() - t) / 2))

===== PROFLOGIT vs BİZİMKİ =====


telco / Bizimki (ham LightGBM + kâr-eşiği): EMPC=0.0504 kâr=110205 ROI=1.34 PR-AUC=0.664 (3s)
telco / ProfLogit (EMPC-maksimize): EMPC=0.0492 kâr=107522 ROI=1.33 PR-AUC=0.655 (3s)


cell2cell: ProfLogit için stratified örneklem n=15000 (tam veri 51047), seed=42.
cell2cell / Bizimki (ham LightGBM + kâr-eşiği): EMPC=0.0350 kâr=148424 ROI=0.70 PR-AUC=0.457 (9s)
cell2cell / ProfLogit (EMPC-maksimize): EMPC=0.0350 kâr=148090 ROI=0.70 PR-AUC=0.378 (9s)


ecommerce / Bizimki (ham LightGBM + kâr-eşiği): EMPC=0.0162 kâr=54072 ROI=1.45 PR-AUC=0.908 (9s)
ecommerce / ProfLogit (EMPC-maksimize): EMPC=0.0064 kâr=21297 ROI=0.22 PR-AUC=0.676 (9s)


iranian / Bizimki (ham LightGBM + kâr-eşiği): EMPC=0.0010 kâr=314 ROI=0.11 PR-AUC=0.958 (10s)
iranian / ProfLogit (EMPC-maksimize): EMPC=-0.0101 kâr=-3008 ROI=-0.69 PR-AUC=0.750 (10s)


bank / Bizimki (ham LightGBM + kâr-eşiği): EMPC=0.0324 kâr=198082 ROI=0.93 PR-AUC=0.706 (6s)
bank / ProfLogit (EMPC-maksimize): EMPC=0.0350 kâr=214021 ROI=1.01 PR-AUC=0.437 (6s)


## 2. Tablo + havuzlanmış anlamlılık

In [3]:
df = pb.tablo(tum)
yaz(S.MSG["bolum"].format(ad="KIYAS TABLOSU (fold ort. ± std)"))
yaz(df.to_string(index=False))
# havuzlanmış (set×fold) Wilcoxon
oe = np.concatenate([tum[k][0]["ours"]["EMPC"] for k in cfg.DATASETS])
pe_pl = np.concatenate([tum[k][0]["proflogit"]["EMPC"] for k in cfg.DATASETS])
ok = np.concatenate([tum[k][0]["ours"]["kar"] for k in cfg.DATASETS])
pk_pl = np.concatenate([tum[k][0]["proflogit"]["kar"] for k in cfg.DATASETS])
pe = wilcoxon(oe, pe_pl).pvalue
pk = wilcoxon(ok, pk_pl).pvalue
yaz(S.MSG8["wilcoxon"].format(n=len(oe), pe=pe, pk=pk))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "profit_baseline_comparison.csv"))

===== KIYAS TABLOSU (fold ort. ± std) =====
Veri Seti                           Yaklaşım             EMPC            Kâr    ROI          PR-AUC  Duyarlılık  Kesinlik    F1  p (EMPC, Wilcoxon)  p (kâr, Wilcoxon)
    telco Bizimki (ham LightGBM + kâr-eşiği)  0.0504 ± 0.0021  110205 ± 3839  1.343 0.6635 ± 0.0223       0.985     0.348 0.514              0.1250             0.1250
    telco         ProfLogit (EMPC-maksimize)  0.0492 ± 0.0027  107522 ± 5178  1.332 0.6554 ± 0.0271       0.981     0.353 0.519              0.1250             0.1250
cell2cell Bizimki (ham LightGBM + kâr-eşiği)  0.0350 ± 0.0010  148424 ± 5542  0.703 0.4572 ± 0.0082       1.000     0.289 0.448              0.2500             0.2500
cell2cell         ProfLogit (EMPC-maksimize)  0.0350 ± 0.0010  148090 ± 5741  0.700 0.3783 ± 0.0127       1.000     0.288 0.448              0.2500             0.2500
ecommerce Bizimki (ham LightGBM + kâr-eşiği)  0.0162 ± 0.0054  54072 ± 18264  1.449 0.9075 ± 0.0160       0.912     0.705

## 3. Figürler

In [4]:
for y in pb.figurler(tum):
    yaz(S.MSG["kayit"].format(yol=y))

Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_baseline/proflogit_vs_ours_empc.png
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_baseline/proflogit_vs_ours_profit.png


## 4. Özet — kâr-eşiği yeterli mi? (karar kullanıcıda)

In [5]:
yaz(S.MSG["bolum"].format(ad="ÖZET"))
for k in cfg.DATASETS:
    m = tum[k][0]
    de = np.mean(m["ours"]["EMPC"]) - np.mean(m["proflogit"]["EMPC"])
    dk = np.mean(m["ours"]["kar"]) - np.mean(m["proflogit"]["kar"])
    kazanan = "bizimki" if de >= 0 else "ProfLogit"
    yaz(f"  {k:11s} EMPC farkı(bizim−PL)={de:+.4f} kâr farkı={dk:+.0f} -> {kazanan} önde")
yaz(f"\nHavuzlanmış: EMPC p={pe:.4f}, kâr p={pk:.4f} "
    f"({'anlamlı' if pe < 0.05 else 'gürültü'} / {'anlamlı' if pk < 0.05 else 'gürültü'})")
yaz(S.MSG8["bitti"])

_log = cfg.LOGS / "adim8_ozet.log"
_log.write_text("\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== ÖZET =====
  telco       EMPC farkı(bizim−PL)=+0.0012 kâr farkı=+2683 -> bizimki önde
  cell2cell   EMPC farkı(bizim−PL)=+0.0001 kâr farkı=+333 -> bizimki önde
  ecommerce   EMPC farkı(bizim−PL)=+0.0098 kâr farkı=+32775 -> bizimki önde
  iranian     EMPC farkı(bizim−PL)=+0.0111 kâr farkı=+3322 -> bizimki önde
  bank        EMPC farkı(bizim−PL)=-0.0026 kâr farkı=-15939 -> ProfLogit önde

Havuzlanmış: EMPC p=0.0129, kâr p=0.1096 (anlamlı / gürültü)
ADIM 8 tamamlandı. Kâr-odaklı baseline kıyaslandı; yorum/karar kullanıcıya bırakıldı.
Kaydedildi: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/logs/adim8_ozet.log
